# DMRG on 2D systems and entropy scaling

<div style="text-align: right; font-style: italic; color: #6b7280; margin: 8px 0; font-size: 0.95em;">
Authored by <strong>Gauthameshwar S.</strong>
</div>

<div style="
    border: 1px solid #3b82f6;
    border-left: 5px solid #3b82f6;
    padding: 12px 14px;
    border-radius: 8px;
    margin: 12px 0;
    box-shadow: 0 1px 2px rgba(0,0,0,0.06);
">
    <div style="font-weight: 700; margin-bottom: 6px;">TL;DR</div>
    <div>
        This notebook applies DMRG to a <strong>2D transverse-field Ising model</strong> by representing the 2D lattice as a 1D MPS (via a <strong>snake ordering</strong>).
        In 2D, the entanglement across a geometric cut grows with the system width, which makes the required MPS bond dimension grow rapidly with lattice size.
    </div>
</div>

### What you will do here
- Define the 2D TFI Hamiltonian and build an MPO using <strong>ITensors.jl</strong>
- Run a small (2×2) DMRG demo and inspect the resulting MPS
- Compute local observables like $\langle S^x_i\rangle$ and $\langle S^z_i\rangle$
- Introduce a snake mapping (2D → 1D ordering) and use it to build nearest‑neighbor bonds
- Compute a bipartite entanglement entropy across a chosen mid‑chain bond that corresponds to a geometric cut in 2D
- Scan lattice sizes and plot the entropy scaling with CairoMakie

## 2D transverse-field Ising model (2×2 demo)

We model a square lattice of spin-$\frac12$ sites with nearest‑neighbor Ising coupling along $x$ and a transverse field along $z$:
$$
H = -J\sum_{\langle i,j\rangle} \sigma_i^x\sigma_j^x - h_z\sum_i \sigma_i^z.
$$
- $\langle i, j \rangle$ corresponds to nearest-neighbor lattice pairs.
- $J$ controls the Ising coupling.
- $h_z$ is the transverse field strength.
<div style="border: 1px solid #f97316; border-left: 5px solid #f97316; padding: 12px 14px; border-radius: 8px; margin: 12px 0; box-shadow: 0 1px 2px rgba(0,0,0,0.06);">
    <div style="font-weight:700; margin-bottom:6px;">Implementation note (ITensors convention).</div>
    <div>
        In the code we use the spin operators $S^\alpha = \sigma^\alpha/2$. That is why the MPO terms use factors like $-4J$ for $S^x_i S^x_j$ and $-2h_z$ for $S^z_i$.
    </div>
</div>

For the 2×2 lattice (4 sites), we build the MPO from an `OpSum` by summing over all nearest-neighbor pairs, and run a short DMRG sweep to get the ground state.

In [ ]:
using ITensors, ITensorMPS

## Snake mapping (2D → 1D) in one page

DMRG works with a 1D ordering of sites, because an MPS is a 1D tensor network. To apply it to a 2D lattice, we choose a map
$$
(x,y) \mapsto n \in \{1,\dots,L_xL_y\}
$$
that linearizes the lattice. The Hamiltonian is still a sum of *2D nearest‑neighbor* terms, but when expressed in the 1D ordering it contains many longer‑range couplings along the chain.

A common choice is a **snake ordering** that traverses one row left→right, the next row right→left, etc.

### Example: 2×2 snake ordering
For $L_x=L_y=2$ the snake path is
$$
(1,1)\to(2,1)\to(2,2)\to(1,2)
$$
so the site labels are
- $(1,1)\mapsto 1$, $(2,1)\mapsto 2$,
- $(2,2)\mapsto 3$, $(1,2)\mapsto 4$.

Nearest-neighbor lattice bonds (horizontal/vertical) become bonds between these 1D labels (some of which are not adjacent along the chain for larger lattices). In the code below, `snake_bonds(Nx,Ny)` explicitly constructs the neighbor pairs $(i,j)$ you can feed into an `OpSum`.

In [ ]:
# Build nearest-neighbor bonds using a snake mapping (1D MPS order)
function snake_bonds(Nx, Ny)
    # Obtain the 1D chain site index of a snake mapping from the the lattice indices 
    function snake_index(x, y, Nx)
        isodd(y) ? (y - 1) * Nx + x : (y - 1) * Nx + (Nx - x + 1)
    end

    bonds = Tuple{Int,Int}[]
    for (x, y) in Iterators.product(1:Nx, 1:Ny)
        i = snake_index(x, y, Nx)
        if x < Nx
            j = snake_index(x + 1, y, Nx)
            push!(bonds, (i, j))
        end
        if y < Ny
            j = snake_index(x, y + 1, Nx)
            push!(bonds, (i, j))
        end
    end
    bonds
end


In [ ]:
snake_bonds(2,2)

In [ ]:
# 2×2 transverse-field Ising model
Nx = 2
Ny = 2
N = Nx * Ny

# Define the 1D spin-1/2 sites of size N

# generate the nearest neighbor in the 1D chain from the 2D mesh 
# you can either use the snake_bonds function or use square_lattice function from ITensorMPS

# Hamiltonian parameters
J = 1.0
h_z = 0.5

# Define H = -J Σ⟨ij⟩ σx_i σx_j - h_z Σ σz_i
# you need to sum over the bonds generated above to define the coupling terms

# Iniaialise the system with all spins pointing in the X+ direction |→→...→⟩

# Compute the inner product ⟨ψ0|H|ψ0⟩ and verify if it matches the expected value ⟨H⟩ = -2L(L-1)J for a 2D square lattice of size LxL

In [ ]:
# Do a dmrg with maxdim 1000 and cutoff 1e-10 to find the ground state energy and wavefunction


In [ ]:
# print the bond dimension of the resulting MPS (assuming `psi` is the ground state MPS from DMRG)
# for b in 1:(N-1)
#     println("Bond $b dimension: ", maximum(dims(psi[b])))
# end


In [ ]:
# local magnetisation of the ground state (operators in the Hamiltonian)
# mx = real(expect(psi, "Sx"))
# my = real(expect(psi, "Sy"))
# mz = real(expect(psi, "Sz"))
# println("Magnetisation at each site:\n <Sx> = $mx,\n <Sy> = $my,\n <Sz> = $mz")


## The 2D bond dimension disaster

For 2D ground states of local Hamiltonians, entanglement typically follows an **area law**, so the entropy scales with the boundary length of a cut: $S \sim N$. At criticality there can be multiplicative logarithmic corrections (often $S \sim N\log N$), but it is still boundary‑law scaling. However, when representing a 2D system as a 1D MPS (snake mapping), this boundary‑law entanglement implies a **bond dimension that grows exponentially with the width**.

Here $N$ is the side length of the square lattice (e.g., $N=4$ for a $4\times4$ lattice).

| Phase | Spectral gap $\Delta$ | Correlation length $\xi$ | Entanglement entropy | Bond dimension scaling (snake‑MPS) | DMRG behavior |
|------|------------------------|--------------------------|----------------------|------------------------------------|---------------|
| 2D Ferromagnetic ($h < 1$) | Finite | Finite ($\xi \sim \Delta^{-1}$) | $S \sim N$ | $\chi \sim e^{cN}$ | Very costly |
| 2D Critical ($h = 1$) | Zero | Divergent | $S \sim N\log N$ (often) | $\chi \sim e^{cN\log N}$ | Very costly |
| 2D Paramagnetic ($h > 1$) | Finite | Finite ($\xi \sim \Delta^{-1}$) | $S \sim N$ | $\chi \sim e^{cN}$ | Very costly |

## Bipartite entropy for a horizontal cut in the 1D snake

For the horizontal bipartition
$$
A = \{(x,y): y \le L_y/2\},\qquad B = \{(x,y): y > L_y/2\},
$$
we expect the entanglement entropy to scale linearly with system size according to the area law. To obtain this value, choose a 1D ordering where **all sites in $A$ come first**, then all sites in $B$.
In the snake ordering, If we cut the bonds at $⌊Ly/2\rfloor Lx$, we have a system where the left part of the chain describes all the sites in the partition $A$, and the right part contained in $B$. 
Compute the bipartite entropy by putting the MPS in mixed‑canonical form at bond $\ell$, taking the Schmidt values $\lambda$, and evaluating
$$
S = -\sum_k \lambda_k^2\log(\lambda_k^2).
$$
This gives the **exact** von Neumann entropy of $\rho_A$ for that geometric cut (the ordering just makes the cut align with one MPS bond).

In [ ]:
# Function to obtain the entanglement entropy across a specific bond
function bond_entropy(psi, b)
    orthogonalize!(psi, b)
    left_link = linkind(psi, b-1)
    left_inds = left_link === nothing ? (siteind(psi, b),) : (left_link, siteind(psi, b))
    U, S, V = svd(psi[b], left_inds)
    s = diag(S)
    p = s .^ 2
    return -sum(pi -> (pi > 0 ? pi * log(pi) : 0.0), p)
end

# Dictionary to store max bond dimensions for different h and system sizes
max_bond_dim = Dict{Float64, Vector{Float64}}()
# The array of transverse fields
hs = [0.5, 1.0, 3.0, 4.0, 5.0]
# The array of system sizes (Nx = Ny = n)
ns = 4:8

# Run a for loop over different h
    # define an array to store the max bond dimensions for different system sizes

    # Run a for loop over different system sizes
        # define the 2D lattice parameters Nx, Ny, N

        # define the spin-1/2 sites

        # define the simple product initial state with all spins in X+ direction

        # define the Hamiltonian H using the snake_bonds function with J = 1.0

        # run a DMRG to find the ground state wavefunction with cutoff 1e-12

        # compute the bond entropies across all bonds
        
        # store the bond entropy across the bond (Nx÷2)Ny in the array
    # end for n
# end for h


In [ ]:
# CairoMakie plot of the bipartite entropy vs lattice size (Uncomment to plot)
# using CairoMakie
# fig = Figure(; size=(600,400))
# ax = Axis(fig[1, 1]; 
#     xlabel="Lattice Size (N x N)", 
#     ylabel=L"S(\rho_A)", 
#     title="Bipartite entropy vs Lattice Size for 2D TFI Model", 
#     )
# for h in hs
#     lines!(ax, ns, max_bond_dim[h], label="h = $h")
#     scatter!(ax, ns, max_bond_dim[h])
# end
# axislegend(ax; position=:lt)
# fig